# 04 Training and evaluation within the flow datasets

The CNN (64 × 64), the chip network (8 × 8) and the feature control are trained and tested within one flow dataset,
with five-fold grouped cross-validation. A group is one recording, so all images of a recording are either in the
training or in the test part. The validation set (epoch selection) consists of whole recordings of the training
part.

* **Andor Zyla flow dataset** (0.65 µm/px), 76 recordings.
* **Phantom flow dataset** (0.925 µm/px), 12 recordings. Bright field and Ph2 are trained together.

Each recording contains only one cell line, and each cell line was recorded on its own days. Single test folds can
therefore contain only one class.

Thesis: Results, paragraphs "Training and evaluation within the flow dataset" in the Sections on the Andor Zyla flow
dataset (Table `tab:res-transfer`) and on the Phantom flow dataset (Table `tab:res-transfer-phantom`).

Output: `results/04_zyla_flow_cv_*` and `results/04_phantom_flow_cv_*`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display

import thesis_config as C     # paths and settings
import thesis_ml as M         # data, preprocessing, models, training and evaluation

pd.set_option('display.precision', 3)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', C.RESULTS_DIR.resolve())
if C.FAST:
    print('QUICK TEST (THESIS_FAST=1): few images and 3 epochs, the numbers are not meaningful.')

## 1. Data

In [ ]:
data = C.load_data(sets=['flow_zyla', 'flow_phantom'])
OUTPUT = {'flow_zyla': '04_zyla_flow_cv', 'flow_phantom': '04_phantom_flow_cv'}
for k, d in data.items():
    print(f'{k:13s} recordings per cell line:', d.meta.groupby('label').group.nunique().to_dict())

## 2. Cross-validation over recordings

In [ ]:
results = {}
for k, d in data.items():
    folds = M.grouped_folds(d.meta.y.to_numpy(), d.meta.group.to_numpy(), C.N_FOLDS, seed=0)
    print(k)
    display(M.split_table(d.meta, 'group', folds).pivot_table(index='fold', columns='part',
            values=['n_groups', 'n_HeLa', 'n_NIH3T3']))
    metrics, preds = [], []
    for kind, size in C.MODELS:
        m, p, _ = M.run_grouped_cv(d.meta, d.X, kind, size, 'group', C.TRAIN, C.N_FOLDS, C.SEEDS_BY_MODEL[kind],
                                   C.VAL_FRAC, scenario=f'{k}_cv')
        metrics.append(m); preds.append(p)
    metrics.append(M.run_control_cv(d.meta, d.feat, 'group', C.N_FOLDS, scenario=f'{k}_cv'))
    results[k] = (pd.concat(metrics, ignore_index=True), pd.concat(preds, ignore_index=True))
    M.save_results(OUTPUT[k], C.RESULTS_DIR, *results[k])

## 3. Results
Macro-F1, mean ± standard deviation over folds and seeds (feature control: over folds). The second table gives the
mean over the seeds for every test fold, which shows how strongly the result depends on the recordings in the fold.

In [ ]:
for k, (m, _) in results.items():
    print(k)
    display(m.groupby('model').macro_f1.agg(['mean', 'std', 'count']).round(3))
    display(m.groupby(['model', 'fold']).macro_f1.mean().unstack('fold').round(3))